# 02 — Feature Engineering

Shows how raw IEEE-CIS columns become the **curated, interpretable feature set** the models
score on, and how the shared sklearn preprocessor encodes them. The curated set is the single
source of truth shared by training, the API, and the SHAP reason codes.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")
import pandas as pd
from fraud.train.dataset import load_raw
from fraud.features.engineering import build_training_frame, prepare_serving_frame
from fraud.data.schema import CURATED_NUMERIC, CURATED_CATEGORICAL, DERIVED_NUMERIC, IEEE_COLUMN_MAP
from fraud.models.preprocess import build_preprocessor

df = load_raw(20_000)
print("raw columns:", df.shape[1])

[info] IEEE-CIS not downloaded; using synthetic data. See data/README.md.
raw columns: 21


## 1. The curated feature contract

In [2]:
print("numeric    :", CURATED_NUMERIC)
print("categorical:", CURATED_CATEGORICAL)
print("derived    :", DERIVED_NUMERIC)
print()
print("raw -> curated mapping:")
pd.Series(IEEE_COLUMN_MAP).to_frame("raw_ieee_column")

numeric    : ['transaction_amt', 'dist1', 'card1', 'card2', 'addr1', 'C1', 'C2', 'C13', 'C14', 'D1', 'D4', 'D15']
categorical: ['product_cd', 'card_network', 'card_type', 'purchaser_email_domain', 'recipient_email_domain', 'device_type']
derived    : ['hour', 'day_of_week']

raw -> curated mapping:


,raw_ieee_column
transaction_amt,TransactionAmt
dist1,dist1
card1,card1
card2,card2
addr1,addr1
C1,C1
C2,C2
C13,C13
C14,C14
D1,D1


## 2. Raw → curated transformation

In [3]:
X, y = build_training_frame(df)
print("curated matrix:", X.shape, "| target rate:", f"{y.mean():.3%}")
X.head(5)

curated matrix: (20000, 20) | target rate: 3.475%


,transaction_amt,dist1,card1,card2,addr1,C1,C2,C13,C14,D1,D4,D15,product_cd,card_network,card_type,purchaser_email_domain,recipient_email_domain,device_type,hour,day_of_week
0,56.69,60.0,13549,281.0,272.0,3.0,3.0,2.0,2.0,73.0,341.0,309.0,c,american express,debit,gmail,aol,mobile,0,5
1,243.65,NaN,10879,141.0,196.0,2.0,1.0,2.0,6.0,58.0,11.0,141.0,c,mastercard,credit,gmail,outlook,mobile,0,5
2,210.02,NaN,13718,412.0,326.0,1.0,0.0,4.0,3.0,426.0,361.0,266.0,w,visa,debit,hotmail,<NA>,desktop,1,5
3,31.15,NaN,10249,244.0,194.0,1.0,2.0,4.0,2.0,552.0,231.0,60.0,w,mastercard,debit,gmail,<NA>,mobile,2,5
4,39.34,57.8,10239,404.0,289.0,0.0,1.0,4.0,2.0,381.0,NaN,538.0,r,visa,debit,gmail,yahoo,mobile,2,5


Note the interpretable categoricals (`purchaser_email_domain` collapsed to its provider
family, lowercase product/network codes) and the derived `hour` / `day_of_week`.

## 3. Encoding via the shared preprocessor

In [4]:
pre = build_preprocessor(scale=True)
Z = pre.fit_transform(X)
print("encoded matrix:", Z.shape, "(one-hot expands the 6 categoricals)")
names = pre.get_feature_names_out()
print("first 12 encoded feature names:")
for n in names[:12]:
    print("  ", n)

encoded matrix: (20000, 43) (one-hot expands the 6 categoricals)
first 12 encoded feature names:
   num__transaction_amt
   num__dist1
   num__card1
   num__card2
   num__addr1
   num__C1
   num__C2
   num__C13
   num__C14
   num__D1
   num__D4
   num__D15


## 4. The serving path produces the same columns

In [5]:
rec = {"transaction_amt": 980.0, "product_cd": "C", "card_network": "VISA",
       "purchaser_email_domain": "anonymous.com", "transaction_dt": "2024-05-01T03:30:00Z",
       "C1": 9, "C13": 14}
sf = prepare_serving_frame(rec)
print("serving frame columns match training:", list(sf.columns) == list(X.columns))
sf.T

serving frame columns match training: True


,0
transaction_amt,980.0
dist1,NaN
card1,NaN
card2,NaN
addr1,NaN
C1,9
C2,NaN
C13,14
C14,NaN
D1,NaN


**Takeaway:** missing categoricals become an explicit `"missing"` category (absence is
signal in fraud), `min_frequency` folds rare levels together, and the identical contract on
both paths means train and serve can never silently diverge.

---
Next: `03_benchmark`.